# Main chatbot notebook — GitHub version

This notebook loads the project directly from the GitHub repository instead of searching for files in Google Drive.

Use it in this order:

1. Clone or update the repository.
2. Install the requirements.
3. Configure the environment.
4. Reload the project modules.
5. Load one model.
6. Initialize and test the chatbot.

To test another model, release the current model, change `MODEL_NAME`, and rerun the LLM loading and chatbot initialization cells.


In [ ]:
# Cell 1 — Clone or update the project from GitHub
from pathlib import Path
import os
import sys
import subprocess
import shutil

REPO_URL = "https://github.com/MessaAlberto/HMD_aquatic_center_chatbot.git"
BRANCH = "main"
PROJECT_DIR = Path("/content/HMD-project")
FORCE_RECLONE = False

if PROJECT_DIR.exists() and FORCE_RECLONE:
    shutil.rmtree(PROJECT_DIR)

if PROJECT_DIR.exists() and (PROJECT_DIR / ".git").exists():
    print("Repository already exists. Pulling latest changes...")
    subprocess.run(["git", "-C", str(PROJECT_DIR), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "checkout", BRANCH], check=True)
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "origin", BRANCH], check=True)
elif PROJECT_DIR.exists():
    raise RuntimeError(
        f"{PROJECT_DIR} already exists but is not a Git repository. "
        "Set FORCE_RECLONE = True and rerun this cell if you want to overwrite it."
    )
else:
    subprocess.run(
        ["git", "clone", "--branch", BRANCH, REPO_URL, str(PROJECT_DIR)],
        check=True,
    )

COLAB_UTILS_DIR = PROJECT_DIR / "colab_utils"
REQUIREMENTS_PATH = COLAB_UTILS_DIR / "requirements_colab.txt"

os.chdir(PROJECT_DIR)

for path in [PROJECT_DIR, COLAB_UTILS_DIR]:
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

os.environ["REQUIREMENTS_PATH"] = str(REQUIREMENTS_PATH)

print("Project folder:", PROJECT_DIR)
print("Current directory:", os.getcwd())
print("Requirements file:", REQUIREMENTS_PATH)


In [ ]:
# Cell 2 — Install/update dependencies
print("Installing requirements from:", REQUIREMENTS_PATH)
%pip install -q -r "$REQUIREMENTS_PATH"


In [ ]:
# Cell 3 — Environment and debug configuration
import os
import logging

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None

if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    print("HF_TOKEN loaded from Colab Secrets.")
else:
    print("HF_TOKEN not found. Public models can still be loaded if access is not restricted.")

APP_DEBUG = False
os.environ["APP_DEBUG"] = str(APP_DEBUG).lower()
os.environ["TOKENIZERS_PARALLELISM"] = "false"

logging.basicConfig(
    level=logging.DEBUG if APP_DEBUG else logging.WARNING,
    format="%(levelname)s:%(message)s",
    force=True,
)

print(f"APP_DEBUG={os.environ['APP_DEBUG']}")


In [ ]:
# Cell 4 — Verify the project structure
from pathlib import Path

required_files = [
    "colab_utils/requirements_colab.txt",
    "app/chatbot.py",
    "llm/config.py",
    "llm/loader.py",
    "llm/generation.py",
    "components/router.py",
    "components/NLU.py",
    "components/DM.py",
    "components/NLG.py",
]

missing = [path for path in required_files if not (PROJECT_DIR / path).exists()]

if missing:
    print("Missing files:")
    for path in missing:
        print("-", path)
    raise FileNotFoundError("The cloned repository does not contain all required files.")

print("All required project files found.")


In [ ]:
# Cell 5 — Reload project modules from the cloned repository
import os
import sys
import importlib
from pathlib import Path

if "PROJECT_DIR" not in globals():
    raise RuntimeError("PROJECT_DIR is not defined. Run Cell 1 first.")

os.chdir(PROJECT_DIR)

for path in [PROJECT_DIR, COLAB_UTILS_DIR]:
    path_str = str(path)
    if path_str not in sys.path:
        sys.path.insert(0, path_str)

Path("app/__init__.py").touch()
importlib.invalidate_caches()

MODULES_TO_REFRESH = [
    "settings",
    "utils.settings",
    "prompts.router_prompt",
    "prompts.nlu_prompt",
    "prompts.dm_prompt",
    "prompts.nlg_prompt",
    "llm.generation",
    "llm.config",
    "llm.loader",
    "components.router",
    "components.NLU",
    "components.DM",
    "components.NLG",
    "state.dialogue_state_tracker",
    "state.history",
    "state.task_queue",
    "database.db_controller",
    "database.mock_database",
    "app.chatbot",
]

for module_name in MODULES_TO_REFRESH:
    if module_name in sys.modules:
        del sys.modules[module_name]

importlib.invalidate_caches()

from app.chatbot import Chatbot
from components.router import Router
from components.NLU import NLU
from components.DM import DM
from components.NLG import NLG
from state.dialogue_state_tracker import StateTracker
from state.history import History
from database.db_controller import DBController
from state.task_queue import TaskQueue
from llm.loader import load_llm

print("Project modules loaded from:", PROJECT_DIR)
print("If you pull new repository changes, rerun this cell.")
print("If you changed llm.loader, llm.config, or llm.generation, reload the LLM too.")


In [ ]:
# Cell 6 — Load one LLM
# Change this value to test another model from llm.config.
MODEL_NAME = "qwen3_4b"

llm = load_llm(MODEL_NAME)

print(f"LLM loaded once: {MODEL_NAME}")


In [ ]:
# Cell 7 — Initialize the chatbot with the already-loaded LLM
def build_bot_from_loaded_llm(llm):
    bot = Chatbot.__new__(Chatbot)

    bot.llm = llm
    bot.router = Router(llm)
    bot.NLU = NLU(llm)
    bot.DM = DM(llm)
    bot.NLG = NLG(llm)

    bot.dst = StateTracker()
    bot.db_controller = DBController(bot.dst)
    bot.history = History()
    bot.task_queue = TaskQueue()

    bot.DONE_STATUSES = ["INFORM", "CONFIRMED", "ABORTED"]

    return bot


bot = build_bot_from_loaded_llm(llm)

print("Bot initialized with the latest loaded project modules.")
print("You can now test with bot.reply(...) or bot.chat_loop().")


In [ ]:
# Cell 8 — Interactive console loop
# Type exit, quit, or stop to end the loop.
bot.chat_loop()


In [ ]:
# Cell 9 — Inspect conversation history
try:
    bot.history.print_full_conversation()
except Exception as error:
    print(f"Could not print history: {error}")


In [ ]:
# Cell 10 — Reset conversation state without reloading the model
bot.reset_state()
print("Conversation state reset. The LLM is still loaded.")


## Testing another model

To test another model:

1. Run `release_llm()` in Cell 7.
2. Change `MODEL_NAME` in Cell 8.
3. Rerun Cells 8, 9, 10, and 11.

If the repository has changed, rerun Cell 1 and then Cell 5 before loading the model again.
